In [18]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [19]:
openGeometry("beam-2.geo")

mat = Material("beam")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Φ = Field([mat], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m, reducedOrder=true);

In [20]:
b = 30
h = 5
E = mat.E
G = mat.μ
Iz = b * h^4 / 12
κ = 2 / 3
A = b * h;

In [21]:
t = tangentVector(U, "beam")
tx, ty, tz = t[1], t[2], t[3]

nx = -ty
ny = tx
nz = 0;

In [22]:
# ------------------------------------------------------------------
# Generalized Timoshenko beam strain
#
# ε = tᵀ ∇u t
# γ = nᵀ ∇u t - φ
# κ = ∇φ ⋅ t
# ------------------------------------------------------------------

Au = [
    tx*tx tx*ty tx*ty ty*ty
    nx*tx nx*ty ny*tx ny*ty
    0 0 0 0
]

Aφ = [
    0 0
    0 0
    tx ty
]

Gφ = [
    0; -1; 0;;
]

Bu = Au ⋅ Grad(U)

Bφ = Aφ ⋅ Grad(Φ) + Gφ ⋅ Φ;

In [23]:
D = [
    E*A 0 0
    0 κ*G*A 0
    0 0 E*Iz
]

3×3 Matrix{Float64}:
 3.0e7  0.0        0.0
 0.0    7.69231e6  0.0
 0.0    0.0        3.125e8

In [24]:
B = Bu + Bφ
K = ∫(B' ⋅ D ⋅ B, Γ="beam");

In [25]:
supp_u = BoundaryCondition("A", field=U, ux=0, uy=0)
supp_φ = BoundaryCondition("A", field=Φ, φ=0);

In [26]:
fu = ∫(U ⋅ [1, 0], Γ="C")
fφ = ∫(Φ ⋅ 0, Γ="B")
fu2 = ∫(U ⋅ [0.0, -0.0], Γ="beam");

In [27]:
F = SystemVector([fu + fu2, fφ]);

In [28]:
u, φ = solveField(K, F, support=[supp_u, supp_φ]);

In [29]:
showDoFResults(fu + fu2, name="force")
u0 = showDoFResults(u, name="u", factor=1000, visible=true)
φ0 = showDoFResults(φ, name="φ");

In [30]:
n = VectorField([nx, ny, 0]);

In [31]:
g = grad(expandTo3D(u))

ε = t ⋅ (g * t)
γ = n ⋅ (g * t) - nodesToElements(φ)

κb = grad(φ) ⋅ t

N = E * A * ε
T = κ * G * A * γ
Mh = E * Iz * κb;

In [32]:
N0 = showElementResults(N, name="N")
T0 = showElementResults(T, name="T")
Mh0 = showElementResults(Mh, name="Mh")

5

In [33]:
plotOnBeam("beam", N, name="N graph")
plotOnBeam("beam", elementsToNodes(N), name="N graph")
plotOnBeam("beam", T, name="T graph")
plotOnBeam("beam", elementsToNodes(T), name="T graph")
plotOnBeam("beam", Mh, name="Mh graph")
plotOnBeam("beam", elementsToNodes(Mh), name="Mh graph")

11

In [34]:
openPostProcessor()